# **CardiAg: ECG & RESP Preprocessing Data Quality Summary of Main Blocks**

Pipeline for extracting relevant features from the ECG and RESP preprocessing specifically related to the main blocks (i.e., pre- and post-task baseline, plus task experimental conditions - BasA, OpA, BasT, OpT - without breaks). 

* For the ECG preprocessing, the following features are computed: percentage of manually corrected R-peaks in the main blocks, percentage of bad segments in the main blocks, heart rate (HR) and RR interval duration in the pre- & post-task baseline, as well as in the main experimental blocks. 
* For the RESP preprocessing, the following features are computed: percentage of manually corrected expiration & inspiration onsets in the main blocks, percentage of bad segments in the main blocks, respiratory rate (RResp) and respiratory cycle duration in the pre- & post-task baseline, as well as in the main experimental blocks. 

Authors: Niket Aggrawal, Marta Gerosa

Created on: 24 January 2025

Last update (by M. Gerosa): 22 April 2026

## **Initial settings**

Specify the participants list, the BIDS-compliant directory and entities, as well as the type of R-peaks to extract from the ECG preprocessed file (e.g., manually corrected R-peaks from "ECG_R_Peaks_ManualCorr") and the sampling frequency. 

In [21]:
############## Import modules ##############

import pandas as pd
import numpy as np
import json
import os
import neurokit2 as nk
from systole.utils import input_conversion
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown as md

In [ ]:
################## Specify BIDS-compatible data information ##################

# Specify the data path info
wd = r'.\data'              # directory of data storage
exp_name = 'CardiAgIBTask'  # current study or experiment name
datatype_name = 'beh'       # current datatype folder according to BIDS

# Specify sampling frequency of peripheral physio recording
sfreq = 1000


##### ECG ##### 
# Specify the included participants list in the cardiac analyses: 
# sub-14 (missing HW triggers,) sub-16 (extreme JE distribution), sub-33 (outlier IB measures)
participant_ids_ecg = [1, 2, 3, 5, 6, 12, 13, 15, 17, 18, 19,
                       20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 
                       31, 32, 34, 35, 36, 37, 38, 39, 41, 42, 
                       43, 44, 45, 46, 47, 48, 49, 51, 53, 54, 55, 57] 

# Set which type of R-peaks to extract from ECG preprocessed file: uncorr, autocorr or manualcorr
rpeaks_var = 'ECG_R_Peaks_ManualCorr'
rr_var = 'RR_s_ManualCorr'


##### RESP ##### 
# Specify the included participants list in the respiratory analyses:
# sub-14 (missing HW triggers,) sub-16 (extreme JE distribution), sub-33 (outlier IB measures)
participant_ids_resp = [1, 2, 3, 5, 6, 12, 13, 15, 17, 18, 19,
                       20, 21, 22, 23, 24, 25, 26, 27, 28, 30,        # removed sub-29 (bad RESP signal quality)
                       31, 32, 34, 35, 36, 37, 38, 39, 41,            # removed sub-42 (bad RESP signal quality)
                       43, 44, 45, 47, 48, 49, 51, 53, 54, 55, 57]    # removed sub-46 (bad RESP signal quality)

# Set which type of expiration/inspiration onsets to extract from RESP preprocessed file: uncorr, manualcorr
resp_onsets_var = 'ManualCorr'

## **1. Data import and ECG features selection**

- **1a. Define function to load preprocessed behavioral & task event triggers**: define function to load subject-specific preprocessed behavioral data (`_beh-preproc.tsv`) from `'derivatives\beh-preproc\sub-XX\beh\'`, and task event triggers (`_events.tsv`) from `'data\sub-XX\beh\'`.
- **1b. Define function to load ECG/RESP preprocessing file**: define function to load subject-specific preprocessed ECG data (`_ecg-preproc.json`) from `'derivatives\ecg-preproc\sub-XX\beh\'`, and preprocessed RESP data (`_resp-preproc.json`) from `'derivatives\resp-preproc\sub-XX\beh\'`.
- **1c. Define functions to extract main ECG/RESP features and bad segments in main blocks**: define functions to extract main features in a the main blocks (baselines + experimental blocks), given the start and end of event is provided. In detail, these include the R-peaks in ECG signal, or expiration/inspiration onsets in RESP signal, as well as the pairs of indices of bad segments included between the start and end timepoint of a given block for both physio modalities. 
- **1d. Segmentation of ECG/RESP features into main blocks (pre- & post-baseline, experimental blocks)**: get condition order for main experimental blocks and onset/offset of each block from behavioral data. Then, proceed to extract the R-peaks, expiration/inspiration onsets and bad segments included in each main block, including pre- and post-baseline, and 2x condition experimental blocks (BasA, OpA, BasT, OpT). All the extracted information is saved within the subject-specific summary JSON file `_ecg-mainblocks.json` and `_resp-mainblocks.json`, respectively. 
- **1e. Run for all participants**: run all the functions above in a loop for each participant included in `participant_ids_ecg` and `participant_ids_resp`, respectively. 

In [23]:
############## 1a. Load behavioral and event triggers ##############

def load_beh_and_events(wd, exp_name, datatype_name, curr_subject):

    # Load behavioral preprocessed data for current subject
    folder_beh = os.path.join(wd, "derivatives", "beh-preproc", curr_subject, datatype_name)
    file_beh = f"{folder_beh}/{curr_subject}_task-{exp_name}_beh-preproc.tsv"
    df_beh = pd.read_csv(file_beh, sep="\t")[
        ["subjID","condition","n_block","n_trial",
         "trial_onset","clock_onset","keypress_onset","tone_onset"]]

    # Load task events for current subject (i.e., start/end event HW triggers)
    folder_events = os.path.join(wd, curr_subject, datatype_name)
    file_events = f"{folder_events}/{curr_subject}_task-{exp_name}_events.tsv"
    df_events = pd.read_csv(file_events, sep="\t")

    return df_beh, df_events


############## 1b. Load ECG/RESP preprocessing .json file ##############

def load_physio_json(wd, exp_name, datatype_name, curr_subject, modality):

    # Load ECG/RESP preprocessed data for current subject
    folder = os.path.join(wd, "derivatives", f"{modality}-preproc", curr_subject, datatype_name)
    file = f"{folder}/{curr_subject}_task-{exp_name}_{modality}-preproc.json"
    
    return json.load(open(file)), folder

In [24]:
################## 1c. Define functions to extract main ECG/RESP features and bad segments in main blocks ##################

# Extract main features in a specific task block, given the start and end of event is provided
# e.g., R-peaks in ECG signal, expiration/inspiration onsets in RESP signal
def extract_points(points, start, end):
    return [p for p in points if start <= p <= end]

##### Bad segments #####

# Normalize ECG/RESP bad segment formats into a list of (start, end) pairs
def parse_bad_segments(badsegm):

    # Handle cases where no bad segments were annotated
    if not badsegm:
        return []

    # RESP format: list of [a,b], [c,d], ...
    if isinstance(badsegm[0], list) or isinstance(badsegm[0], tuple):
        # Already paired as required
        return [tuple(pair) for pair in badsegm]

    # ECG format: flat list [a, b, c, d, ...]
    if isinstance(badsegm[0], (int, float)):

        # Handle cases where bad segment indices are odd number 
        if len(badsegm) % 2 != 0:
            print("Warning: ECG bad segment indices MUST be in even numbers")
        # Extract pairs of bad segment indices (even and odd positions)
        return list(zip(badsegm[::2], badsegm[1::2]))

    raise ValueError("Error: unknown bad_segments format")

# Define whether the start/end event fall within a bad segment or between bad segments
def where_bad_segments(pairs, to_search):
    for i in range(len(pairs)-1):
        bs_curr_on, bs_curr_off = pairs[i]
        bs_next_on, bs_next_off = pairs[i+1]

        if bs_curr_on <= to_search < bs_curr_off: 
            return ("Within", pairs[i], to_search)   # index within bad-segment pairs
        if bs_next_on <= to_search < bs_next_off: 
            return ("Within", pairs[i+1], to_search) # index within bad-segment pairs
        if bs_curr_off <= to_search <= bs_next_on: 
            return ("Between", pairs[i], pairs[i+1], to_search) # index between bad-segment pairs
        if i == len(pairs)-2:
            return ("NoBadSegment", to_search)  # no bad segments

# Extract all bad segments included between start and end of an event
def bad_segments_in_event(pairs, start, end):
    bad_segm = []
    for bs_on, bs_off in pairs:
        if end < bs_on: continue
        if start > bs_off: continue

        # Append bad segments indices, either cut off by start/end or as they are
        bad_segm.append((max(start, bs_on), min(end, bs_off)))
    
    # print("Bad segments in this event :",bad_segm)
    return bad_segm

In [25]:
################## 1d. Segmentation of ECG/RESP features into main blocks (pre- & post-baseline, experimental blocks) ##################

# Function to extract the condition order 
def get_condition_order(df_beh):
    cond_series = df_beh["condition"]
    
    # Find indices in behavioral data where condition changes
    change_indices = cond_series.ne(cond_series.shift()).to_numpy().nonzero()[0]
    cond_order = cond_series.iloc[change_indices].tolist()
    
    # Map each condition to its rank
    cond_rank = {condition: i + 1 for i, condition in enumerate(cond_order)}
    
    return cond_order, cond_rank

# Generic segmentation of within-condition blocks in the experimental task (BasA, OpA, BasT, OpT) from beh data
def get_block_intervals(df_beh):
    
    # Recover condition order
    cond_order, _ = get_condition_order(df_beh)
    blocks = [1, 2]
    intervals = {}
    print(f"Condition order: {cond_order}")

    for cond in cond_order:
        df_cond = df_beh[df_beh.condition == cond]

        end_col = "keypress_onset" if cond == "BasA" else "tone_onset"

        # Compute start and end of main experimental blocks, excl. breaks between them
        for blk in blocks:
            df_blk = df_cond[df_cond.n_block == blk]

            start = 1000 * df_blk[df_blk.n_trial == 1].trial_onset.values[0]    # block onset of 1st trial
            end = df_blk[df_blk.n_trial == 30][end_col].values[0]   # block onset of 30th trial, depending on condition event

            if np.isnan(end):
                end = df_blk[df_blk.n_trial == 30]['clock_onset'].values[0] + (2.56 * 3)    # NoResp trials

            end = 1000 * (end + 6)  # + 6 seconds after 30th trial

            intervals[f"{cond}_{blk}"] = (start, end)
    return intervals


# Extract relevant ECG/RESP features within each main block (pre- & post-baseline, experimental blocks)
def extract_features_per_block(df_beh, df_events, modality_json, folder_out, rpeaks_var=None, resp_var=None):

    # Identify baseline intervals (pre- & post-)
    df_base = df_events[df_events.trial_type.isin(["StartECGBase", "EndECGBase"])]
    pre_start, pre_end, post_start, post_end = df_base.onset.values * 1000

    # Prepare output container (for JSON file)
    out = {}

    ##### ECG #####
    if rpeaks_var:
        rpeaks = modality_json["rpeaks"][rpeaks_var]
        badseg = parse_bad_segments(modality_json["bad_segments"])

        out["pre_baseline"] = {
            "rpeaks": extract_points(rpeaks, pre_start, pre_end),       # extract R-peaks in pre-baseline
            "bad_segments": bad_segments_in_event(badseg, pre_start, pre_end),      # extract bad segments
            "start_end_idx": [pre_start, pre_end],
        }

        out["post_baseline"] = {
            "rpeaks": extract_points(rpeaks, post_start, post_end),     # extract R-peaks in post-baseline
            "bad_segments": bad_segments_in_event(badseg, post_start, post_end),    # extract bad segments
            "start_end_idx": [post_start, post_end],
        }

    ##### RESP #####
    if resp_var:
        exp_on = modality_json["exp_onsets"][resp_var]
        insp_on = modality_json["insp_onsets"][resp_var]
        badseg = parse_bad_segments(modality_json["bad_segments"])

        out["pre_baseline"] = {
            "exp_onsets": extract_points(exp_on, pre_start, pre_end),       # extract expiration onsets in pre-baseline
            "insp_onsets": extract_points(insp_on, pre_start, pre_end),     # extract inspiration onsets in pre-baseline
            "bad_segments": bad_segments_in_event(badseg, pre_start, pre_end),    # extract bad segments
            "start_end_idx": [pre_start, pre_end],
        }

        out["post_baseline"] = {
            "exp_onsets": extract_points(exp_on, post_start, post_end),     # extract expiration onsets in post-baseline
            "insp_onsets": extract_points(insp_on, post_start, post_end),   # extract inspiration onsets in post-baseline
            "bad_segments": bad_segments_in_event(badseg, post_start, post_end),   # extract bad segments
            "start_end_idx": [post_start, post_end],
        }

    # Extract main experimental blocks (BasA, OpA, BasT, OpT)
    intervals = get_block_intervals(df_beh)

    for blk, (start, end) in intervals.items():

        if rpeaks_var:
            out[blk] = {
                "rpeaks": extract_points(rpeaks, start, end),       # extract R-peaks in experimental block
                "bad_segments": bad_segments_in_event(badseg, start, end),  # extract bad segments
                "start_end_idx": [start, end],
            }

        if resp_var:
            out[blk] = {
                "exp_onsets": extract_points(exp_on, start, end),   # extract expiration onsets in experimental block
                "insp_onsets": extract_points(insp_on, start, end), # extract inspiration onsets in experimental block
                "bad_segments": bad_segments_in_event(badseg, start, end),  # extract bad segments
                "start_end_idx": [start, end],
            }

    # Save output data as JSON file
    with open(folder_out, "w") as f:
        json.dump(out, f, indent=4)

    print(f"Data saved to JSON in: {folder_out}")

In [ ]:
################## 1e. Run for all participants ##################

for participant in sorted(set(participant_ids_ecg + participant_ids_resp)):
    curr_subject = f"sub-{participant}"
    print(f"\n---  Processing {curr_subject}  ---")

    df_beh, df_events = load_beh_and_events(wd, exp_name, datatype_name, curr_subject)

    ##### ECG #####
    if participant in participant_ids_ecg:

        # Define the ECG data output directory ('derivatives\ecg-preproc\sub-XX\beh') for current subject 
        ecg_json, ecg_folder = load_physio_json(wd, exp_name, datatype_name, curr_subject, "ecg")
        ecg_outfile = f"{ecg_folder}/{curr_subject}_task-{exp_name}_ecg-mainblocks.json"

        extract_features_per_block(
            df_beh=df_beh,
            df_events=df_events,
            modality_json=ecg_json,     # ECG
            folder_out=ecg_outfile,
            rpeaks_var=rpeaks_var,
        )

    ##### RESP #####
    if participant in participant_ids_resp:

        # Define the RESP data output directory ('derivatives\resp-preproc\sub-XX\beh') for current subject 
        resp_json, resp_folder = load_physio_json(wd, exp_name, datatype_name, curr_subject, "resp")
        resp_outfile = f"{resp_folder}/{curr_subject}_task-{exp_name}_resp-mainblocks.json"

        extract_features_per_block(
            df_beh=df_beh,
            df_events=df_events,
            modality_json=resp_json,    # RESP
            folder_out=resp_outfile,
            resp_var=resp_onsets_var,
        )

## **2. Summary of ECG preprocessing features in main blocks** 

In [27]:
################## 2a. Specify the number of manually corrected R-peaks per subject ##################

# Report below only the number of manually corrected R-peaks; if not reported, the value is assumed to be zero
manualcorr_dict = {'sub-1': 12, 'sub-3': 1, 'sub-5': 1, 
                   'sub-30': 1, 'sub-43': 2, 'sub-46': 1, 'sub-47': 6}

In [28]:
################## 2b. Compute summary of ECG preprocessing features in main blocks ##################

# Initialize the DataFrame for ECG data summary
summ_ecg = []

for participant in participant_ids_ecg:
    curr_subject = f"sub-{str(participant)}"

    # Load the main-block ECG summary file
    folder = os.path.join(wd, "derivatives", "ecg-preproc", curr_subject, datatype_name)
    fname_out = f"{folder}\\{curr_subject}_task-{exp_name}_ecg-mainblocks.json"
    df = json.load(open(fname_out))
    
    tot_rpeaks = 0      # initialize total # of R-peaks
    tot_len_idx = 0     # initialize total length of main blocks
    tot_badsegm_idx = 0 # initialize total length of bad segments included in main blocks
    
    for blk in df.keys():

        # Update total # of R-peaks
        rpeak_n = len(df[blk]['rpeaks'])
        tot_rpeaks += rpeak_n

        # Update total length of main blocks
        len_idx = df[blk]['start_end_idx']
        len_idx = len_idx[1] - len_idx[0]
        tot_len_idx += len_idx

        # Update total length of bad segments included in main blocks
        tot_bs_idx_blk = 0
        for bs in df[blk]['bad_segments']:
            tot_bs_idx_blk += bs[1]-bs[0]
        tot_badsegm_idx += tot_bs_idx_blk
    
    # Compute the total number and percentage of manually corrected R-peaks over the total number of R-peaks of main blocks
    if curr_subject in manualcorr_dict.keys():
        tot_manualcorr = manualcorr_dict.get(curr_subject, 0)
    else: 
        tot_manualcorr = 0
    pct_manualcorr = round(tot_manualcorr*100/tot_rpeaks,3)

    # Compute the percentage of bad segment duration over the total duration of main blocks
    pct_badsegm = round(tot_badsegm_idx*100/tot_len_idx,3)

    ##### Pre-baseline #####
    # Pre-baseline: compute HR (in bpm) and mean RR duration
    prebl_nbeats = len(df['pre_baseline']['rpeaks'])    # number of R-peaks in pre-baseline
    prebl_ms = df['pre_baseline']['start_end_idx'][-1] - df['pre_baseline']['start_end_idx'][0]
    prebl_hr_bpm = round((prebl_nbeats*60)/(prebl_ms/1000),2)   # HR in pre-baseline
    prebl_rpeaks = df['pre_baseline']['rpeaks']
    prebl_rr = [x - prebl_rpeaks[i - 1] for i, x in enumerate(prebl_rpeaks)][1:]
    prebl_avg_rr = sum(prebl_rr)/len(prebl_rr)
    prebl_rr_s = round(prebl_avg_rr/sfreq,3)    # average RR interval duration in pre-baseline

    # Pre-baseline: compute HRV (RMSSD)
    prebl_rrs_dict = {
        'RRI': input_conversion(prebl_rpeaks, input_type='peaks_idx', output_type='rr_ms', sfreq=sfreq),   # RR intervals in miliseconds
        'RRI_Time': np.cumsum(prebl_rr)    # Timestamps in seconds
    }
    prebl_hrv = nk.hrv_time(prebl_rrs_dict)
    prebl_hrv_rmssd = prebl_hrv['HRV_RMSSD'][0]

    # Post-baseline: compute HR (in bpm) and mean RR duration
    postbl_nbeats = len(df['post_baseline']['rpeaks'])
    postbl_ms = df['post_baseline']['start_end_idx'][-1] - df['post_baseline']['start_end_idx'][0]
    postbl_hr_bpm = round((postbl_nbeats*60)/(postbl_ms/1000),2)
    postbl_rpeaks = df['post_baseline']['rpeaks']
    postbl_rr = [x - postbl_rpeaks[i - 1] for i, x in enumerate(postbl_rpeaks)][1:]
    postbl_avg_rr = sum(postbl_rr)/len(postbl_rr)
    postbl_rr_s = round(postbl_avg_rr/sfreq,3)

    # Post-baseline: compute HRV (RMSSD)
    postbl_rrs_dict = {
        'RRI': input_conversion(postbl_rpeaks, input_type='peaks_idx', output_type='rr_ms', sfreq=sfreq),   # RR intervals in miliseconds
        'RRI_Time': np.cumsum(postbl_rr)    # Timestamps in seconds
    }
    postbl_hrv = nk.hrv_time(postbl_rrs_dict)
    postbl_hrv_rmssd = postbl_hrv['HRV_RMSSD'][0]

    # print(curr_subject, tot_rpeaks, tot_manualcorr, pct_manualcorr, tot_len_idx, tot_badsegm_idx, pct_badsegm, prebl_hr_bpm, postbl_hr_bpm, prebl_rr_s, postbl_rr_s, "\n")
    
    # Save summary data in summ_ecg
    summ_ecg.append(
        { 'subjID': curr_subject,
          'total_n_rpeaks': tot_rpeaks, 'total_manualcorr_rpeaks': tot_manualcorr, 'pct_manualcorr_rpeaks': pct_manualcorr,
          'len_mainblocks_idx': round(tot_len_idx,1), 'len_badsegm_idx': round(tot_badsegm_idx), 'pct_badsegm': pct_badsegm,
          'hr_bpm_pre': prebl_hr_bpm, 'hr_bpm_post': postbl_hr_bpm,
          'rr_s_pre': prebl_rr_s, 'rr_s_post': postbl_rr_s, 
          'hrv_rmssd_pre': round(prebl_hrv_rmssd,2), 'hrv_rmssd_post': round(postbl_hrv_rmssd,2)
        })

summ_ecg = pd.DataFrame(summ_ecg)
 
# Define directory of storage of summary data and save TSV file 
summ_df_fname = f"task-{exp_name}_ecg-mainblocks_summary.tsv"
summ_df_path =  os.path.join(wd, "derivatives", "ecg-preproc", summ_df_fname)
summ_ecg.to_csv(summ_df_path, index=False, sep='\t', na_rep="n/a")

In [29]:
# Group-level descriptives: ECG data quality
md("On average, {}% of all R-peaks were manually corrected in the main blocks (i.e., baseline and experimental task, excluding breaks and instructions). On average, {}% of the ECG recording was annotated as being noisy or low-quality, but without major effects on R-peak detection.".format(
    round(summ_ecg['pct_manualcorr_rpeaks'].mean(),2), round(summ_ecg['pct_badsegm'].mean(),2)
))

On average, 0.01% of all R-peaks were manually corrected in the main blocks (i.e., baseline and experimental task, excluding breaks and instructions). On average, 1.53% of the ECG recording was annotated as being noisy or low-quality, but without major effects on R-peak detection.

In [30]:
# Group-level descriptives: HR and RR intervals at rest
md("Mean heart rate at rest, averaged across pre- and post-baseline, varied from {} to {} bpm (M={}, SD={}), with mean RR intervals ranging from {} to {} s (M={}, SD={}).".format(
    round(summ_ecg[['hr_bpm_pre', 'hr_bpm_post']].min().min(),2), round(summ_ecg[['hr_bpm_pre', 'hr_bpm_post']].max().max(),2), 
    round(((summ_ecg['hr_bpm_pre'].mean() + summ_ecg['hr_bpm_post'].mean())/2),2), 
    round(((summ_ecg['hr_bpm_pre'].std() + summ_ecg['hr_bpm_post'].std())/2),2), 
    round(summ_ecg[['rr_s_pre', 'rr_s_post']].min().min(),2), round(summ_ecg[['rr_s_pre', 'rr_s_post']].max().max(),2), 
    round(((summ_ecg['rr_s_pre'].mean() + summ_ecg['rr_s_post'].mean())/2),2), 
    round(((summ_ecg['rr_s_pre'].std() + summ_ecg['rr_s_post'].std())/2),2)
))

Mean heart rate at rest, averaged across pre- and post-baseline, varied from 41.67 to 96.67 bpm (M=72.67, SD=11.61), with mean RR intervals ranging from 0.62 to 1.45 s (M=0.85, SD=0.16).

In [31]:
# Group-level descriptives: RMSSD at rest
md("Mean Root Mean Square of the Successive Differences (RMSSD) at rest, averaged across pre- and post-baseline, varied from {} to {} ms (M={}, SD={}).".format(
    round(summ_ecg[['hrv_rmssd_pre', 'hrv_rmssd_post']].min().min(),2), 
    round(summ_ecg[['hrv_rmssd_pre', 'hrv_rmssd_post']].max().max(),2), 
    round(((summ_ecg['hrv_rmssd_pre'].mean() + summ_ecg['hrv_rmssd_post'].mean())/2),2), 
    round(((summ_ecg['hrv_rmssd_pre'].std() + summ_ecg['hrv_rmssd_post'].std())/2),2)
))

Mean Root Mean Square of the Successive Differences (RMSSD) at rest, averaged across pre- and post-baseline, varied from 9.81 to 161.49 ms (M=41.36, SD=26.87).

## **3. Plotting of average HR and RR duration**

In [32]:
################## 3a. Save df for plotting average HR and RR duration per block ##################

# Initialize list for storing plotting df 
plot_df = []

for participant in participant_ids_ecg:
    curr_subject = "sub-"+str(participant)
    # print("Running for -",curr_subject)

    # 
    folder_out = os.path.join(wd, "derivatives", "ecg-preproc", curr_subject, datatype_name)
    fname_out = f"{folder_out}\\{curr_subject}_task-{exp_name}_ecg-mainblocks.json"
    df = json.load(open(fname_out))

    subj_rr = {}
    subj_hr = {}
    subj_hrv = {}

    for blk in df.keys():
        rpeaks = df[blk]['rpeaks']

        # Convert peaks indices into RR intervals (in seconds) and calculate average HR and RR duration per block
        rr_s = input_conversion(rpeaks, input_type="peaks_idx", output_type="rr_s", sfreq=sfreq)
        avg_rr_s = round(sum(rr_s)/len(rr_s),3)
        avg_hr_bpm = round(60/avg_rr_s,3)
        subj_rr[blk] = avg_rr_s
        subj_hr[blk] = avg_hr_bpm

        # Compute the HRV (RMSSD)
        rrs_dict = {
            'RRI': input_conversion(rpeaks, input_type='peaks_idx', output_type='rr_ms', sfreq=sfreq),   # RR intervals in miliseconds
            'RRI_Time': np.cumsum(rr_s)    # Timestamps in seconds
        }
        time_hrv = nk.hrv_time(rrs_dict)
        hrv_rmssd = time_hrv['HRV_RMSSD'][0]
        subj_hrv[blk] = hrv_rmssd

    plot_df.append(
    {'subjID': curr_subject,
     'preBL_rr': subj_rr['pre_baseline'],
     'BasA1_rr': subj_rr['BasA_1'], 'BasA2_rr': subj_rr['BasA_2'],
     'OpA1_rr': subj_rr['OpA_1'], 'OpA2_rr': subj_rr['OpA_2'], 
     'BasT1_rr': subj_rr['BasT_1'], 'BasT2_rr': subj_rr['BasT_2'], 
     'OpT1_rr': subj_rr['OpT_1'], 'OpT2_rr': subj_rr['OpT_2'],
     'postBL_rr':  subj_rr['post_baseline'],

     'preBL_hr': subj_hr['pre_baseline'],
     'BasA1_hr': subj_hr['BasA_1'], 'BasA2_hr': subj_hr['BasA_2'],
     'OpA1_hr': subj_hr['OpA_1'], 'OpA2_hr': subj_hr['OpA_2'], 
     'BasT1_hr': subj_hr['BasT_1'], 'BasT2_hr': subj_hr['BasT_2'], 
     'OpT1_hr': subj_hr['OpT_1'], 'OpT2_hr': subj_hr['OpT_2'],
     'postBL_hr':  subj_hr['post_baseline'],

     'preBL_rmssd': subj_hrv['pre_baseline'],
     'BasA1_rmssd': subj_hrv['BasA_1'], 'BasA2_rmssd': subj_hrv['BasA_2'],
     'OpA1_rmssd': subj_hrv['OpA_1'], 'OpA2_rmssd': subj_hrv['OpA_2'], 
     'BasT1_rmssd': subj_hrv['BasT_1'], 'BasT2_rmssd': subj_hrv['BasT_2'], 
     'OpT1_rmssd': subj_hrv['OpT_1'], 'OpT2_rmssd': subj_hrv['OpT_2'],
     'postBL_rmssd':  subj_hrv['post_baseline']
    })


plot_df = pd.DataFrame(plot_df)

# Calculate the average RR duration per condition (i.e., merging block 1 & 2) and across all experimental blocks
plot_df['BasA_rr'] = round((plot_df['BasA1_rr'] + plot_df['BasA2_rr'])/2, 3)
plot_df['OpA_rr'] = round((plot_df['OpA1_rr'] + plot_df['OpA2_rr'])/2, 3)
plot_df['BasT_rr'] = round((plot_df['BasT1_rr'] + plot_df['BasT2_rr'])/2, 3)
plot_df['OpT_rr'] = round((plot_df['OpT1_rr'] + plot_df['OpT2_rr'])/2, 3)
plot_df['main_rr'] = round( (plot_df['BasA_rr']+plot_df['OpA_rr']+plot_df['BasT_rr']+plot_df['OpT_rr'])/4 ,3)

# Calculate the average HR (in bpm) per condition (i.e., merging block 1 & 2) and across all experimental blocks
plot_df['BasA_hr'] = round((plot_df['BasA1_hr'] + plot_df['BasA2_hr'])/2, 3)
plot_df['OpA_hr'] = round((plot_df['OpA1_hr'] + plot_df['OpA2_hr'])/2, 3)
plot_df['BasT_hr'] = round((plot_df['BasT1_hr'] + plot_df['BasT2_hr'])/2, 3)
plot_df['OpT_hr'] = round((plot_df['OpT1_hr'] + plot_df['OpT2_hr'])/2, 3)
plot_df['main_hr'] = round( (plot_df['BasA_hr']+plot_df['OpA_hr']+plot_df['BasT_hr']+plot_df['OpT_hr'])/4 ,3)

# Calculate the average RMSSD per condition (i.e., merging block 1 & 2) and across all experimental blocks
plot_df['BasA_rmssd'] = round((plot_df['BasA1_rmssd'] + plot_df['BasA2_rmssd'])/2, 3)
plot_df['OpA_rmssd'] = round((plot_df['OpA1_rmssd'] + plot_df['OpA2_rmssd'])/2, 3)
plot_df['BasT_rmssd'] = round((plot_df['BasT1_rmssd'] + plot_df['BasT2_rmssd'])/2, 3)
plot_df['OpT_rmssd'] = round((plot_df['OpT1_rmssd'] + plot_df['OpT2_rmssd'])/2, 3)
plot_df['main_rmssd'] = round( (plot_df['BasA_rmssd']+plot_df['OpA_rmssd']+plot_df['BasT_rmssd']+plot_df['OpT_rmssd'])/4 ,3)

# # Define directory of storage of plotting df and save as TSV file 
# plots_df_fname = f"{exp_name}_ecg-mainblocks_plots.tsv"
# plots_df_path =  os.path.join(wd, "derivatives", "ecg-preproc", plots_df_fname)
# plot_df.to_csv(plots_df_path, index=False, sep='\t', na_rep="n/a")

In [ ]:
################## 3b. Plot average RR per block & across pre-base, main task and post-base ##################

fig, axs = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [2, 3]}, dpi=300)

color_order = ['lightgrey', '#a4cbb6', '#007a4e', '#7796cb' , '#0b3142', 'lightgrey']
color_order_avg = ['lightgrey', 'chocolate', 'lightgrey']

# Left: plot average RR in pre-bl, main task & post-bl
df = plot_df[['preBL_rr','main_rr', 'postBL_rr']]
axs[0].set_ylim([0.5, 1.5])
sns.boxplot(data=df, ax=axs[0], palette=color_order_avg, saturation=0.8)
sns.stripplot(data=df, color="0.3", size=5, jitter=True, alpha=0.6, ax=axs[0])
axs[0].set_ylabel('Mean RR Interval (s)')
axs[0].set_xticks(['preBL_rr','main_rr', 'postBL_rr'], labels=['Pre-baseline','Main task', 'Post-baseline'])

# Right: plot average RR in pre-bl, BasA, OpA, BasT, OpT & post-bl
df = plot_df[['preBL_rr','BasA_rr', 'OpA_rr', 'BasT_rr', 'OpT_rr', 'postBL_rr']]
axs[1].set_ylim([0.5, 1.5])
sns.boxplot(data=df, ax=axs[1], palette=color_order, saturation=0.8)
sns.stripplot(data=df, color="0.3", size=5, jitter=True, alpha=0.6, ax=axs[1])
axs[1].set_ylabel('Mean RR Interval (s)')
axs[1].set_xticks(['preBL_rr','BasA_rr', 'OpA_rr', 'BasT_rr', 'OpT_rr', 'postBL_rr'], 
                  labels=['Pre-baseline','BasA', 'OpA', 'BasT', 'OpT', 'Post-baseline'])

plt.tight_layout()
sns.despine()
# plt.savefig('avg_rrs.png', dpi=300)
plt.show()

In [ ]:
################## 3c. Plot average HR per block & across pre-base, main task and post-base ##################

fig, axs = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [2, 3]}, dpi=300)

# Left: plot average HR in pre-bl, main task & post-bl
df = plot_df[['preBL_hr','main_hr', 'postBL_hr']]
axs[0].set_ylim([40,110])
sns.boxplot(data=df, ax=axs[0], palette=color_order_avg, saturation=0.8)
sns.stripplot(data=df, color="0.3", size=5, jitter=True, alpha=0.6, ax=axs[0])
axs[0].set_ylabel('Mean HR (bpm)')
axs[0].set_xticks(['preBL_hr','main_hr', 'postBL_hr'], labels=['Pre-baseline','Main task', 'Post-baseline'])

# Right: plot average HR in pre-bl, BasA, OpA, BasT, OpT & post-bl
df = plot_df[['preBL_hr','BasA_hr', 'OpA_hr', 'BasT_hr', 'OpT_hr', 'postBL_hr']]
axs[1].set_ylim([40,110])
sns.boxplot(data=df, ax=axs[1], palette=color_order, saturation=0.8)
sns.stripplot(data=df, color="0.3", size=5, jitter=True, alpha=0.6, ax=axs[1])
axs[1].set_ylabel('Mean HR (bpm)')
axs[1].set_xticks(['preBL_hr','BasA_hr', 'OpA_hr', 'BasT_hr', 'OpT_hr', 'postBL_hr'], 
                  labels=['Pre-baseline','BasA', 'OpA', 'BasT', 'OpT', 'Post-baseline'])

plt.tight_layout()
sns.despine()
# plt.savefig('avg_hr.png', dpi=300)
plt.show()

In [ ]:
################## 3e. Plot average HRV (RMSSD) per block & across pre-base, main task and post-base ##################

fig, axs = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [2, 3]}, dpi=300)

# Left: plot average RMSSD in pre-bl, main task & post-bl
df = plot_df[['preBL_rmssd','main_rmssd', 'postBL_rmssd']]
axs[0].set_ylim([0,150])
sns.boxplot(data=df, ax=axs[0], palette=color_order_avg, saturation=0.8)
sns.stripplot(data=df, color="0.3", size=5, jitter=True, alpha=0.6, ax=axs[0])
axs[0].set_ylabel('Mean RMSSD (ms)')
axs[0].set_xticks(['preBL_rmssd','main_rmssd', 'postBL_rmssd'], labels=['Pre-baseline','Main task', 'Post-baseline'])

# Right: plot average RMSSD in pre-bl, BasA, OpA, BasT, OpT & post-bl
df = plot_df[['preBL_rmssd','BasA_rmssd', 'OpA_rmssd', 'BasT_rmssd', 'OpT_rmssd', 'postBL_rmssd']]
axs[1].set_ylim([0,150])
sns.boxplot(data=df, ax=axs[1], palette=color_order, saturation=0.8)
sns.stripplot(data=df, color="0.3", size=5, jitter=True, alpha=0.6, ax=axs[1])
axs[1].set_ylabel('Mean RMSSD (ms)')
axs[1].set_xticks(['preBL_rmssd','BasA_rmssd', 'OpA_rmssd', 'BasT_rmssd', 'OpT_rmssd', 'postBL_rmssd'], 
                  labels=['Pre-baseline','BasA', 'OpA', 'BasT', 'OpT', 'Post-baseline'])

plt.tight_layout()
sns.despine()
plt.show()

## **4. Data import and RESP features selection**

In [34]:
# Report below only the number of manually corrected expiration/inspiration peaks
resp_manualcorr_dict = {'sub-1': 6, 'sub-2': 20, 'sub-3': 13, 'sub-5': 84, 'sub-6': 22, 'sub-12': 22,
                        'sub-13': 53, 'sub-15': 51, 'sub-17': 42, 'sub-18': 20, 'sub-19': 30, 'sub-20': 18,
                        'sub-21': 2, 'sub-22': 47, 'sub-23': 36, 'sub-24': 23, 'sub-25': 33, 'sub-26': 36, 
                        'sub-27': 2, 'sub-28': 8, 'sub-30': 26, 'sub-31': 24, 'sub-32': 4, 'sub-34': 10, 
                        'sub-35': 74, 'sub-36': 3, 'sub-37': 45, 'sub-38': 35, 'sub-39': 2, 'sub-41': 26, 
                        'sub-43': 54, 'sub-44': 74, 'sub-45': 0, 'sub-47': 39, 'sub-48': 10, 'sub-49': 11,
                        'sub-51': 2, 'sub-53': 0, 'sub-54': 20, 'sub-55': 8, 'sub-57': 2}

In [35]:
################## 4a. Compute summary of RESP preprocessing features in main blocks ##################

# Initialize the DataFrame for RESP data summary
summ_resp = []

for participant in participant_ids_resp:
    curr_subject = f"sub-{str(participant)}"

    # Load the main-block RESP summary file
    mb_folder = os.path.join(wd, "derivatives", "resp-preproc", curr_subject, datatype_name)
    mb_fname = f"{curr_subject}_task-{exp_name}_resp-mainblocks.json"
    mb = json.load(open(os.path.join(mb_folder, mb_fname)))

    tot_exp_onsets = 0
    tot_insp_onsets = 0
    tot_resp_onsets = 0
    tot_len_idx = 0
    tot_badsegm_idx = 0

    for blk, blkdata in mb.items():

        # Update total # of expiration/inspiration onsets
        tot_exp_onsets += len(blkdata.get("exp_onsets", []))
        tot_insp_onsets += len(blkdata.get("insp_onsets", []))
        tot_resp_onsets = tot_exp_onsets + tot_insp_onsets

        # Update total length of main blocks
        start, end = blkdata["start_end_idx"]
        tot_len_idx += (end - start)

        # Update total length of bad segments included in main blocks
        tot_badsegm_idx += sum(
            (bs[1] - bs[0]) for bs in blkdata.get("bad_segments", []))

    # Compute the total number and percentage of manually corrected expiration over the total number of R-peaks of main blocks
    if curr_subject in resp_manualcorr_dict.keys():
        n_corr_total = resp_manualcorr_dict.get(curr_subject, 0)
    else: 
        n_corr_total = 0
    pct_manualcorr_resp = round(n_corr_total*100 / tot_resp_onsets,3) if tot_resp_onsets > 0 else float("nan")
    

    # Compute the percentage of bad segment duration over the total duration of main blocks\
    pct_badsegm_resp = round(tot_badsegm_idx * 100 / tot_len_idx, 3)

    ##### Pre-baseline #####
    # Pre-baseline: compute respiratory rate (RR; in bpm) and mean respiratory cycle duration
    pre = mb["pre_baseline"]
    pre_peaks = pre.get("exp_onsets", [])
    pre_troughs = pre.get("insp_onsets", [])
    pre_ms = pre["start_end_idx"][1] - pre["start_end_idx"][0]

    pre_cycles = min(len(pre_peaks), len(pre_troughs))
    rr_bpm_pre = round((pre_cycles * 60) / (pre_ms / 1000), 2) if pre_cycles > 0 else float("nan")

    # Calculate respiratory cycle duration
    if len(pre_peaks) > 1:
        pre_intervals = np.diff(pre_peaks)
    elif len(pre_troughs) > 1:
        pre_intervals = np.diff(pre_troughs)
    else:
        pre_intervals = []

    cycle_s_pre = round(np.mean(pre_intervals)/1000, 3) if len(pre_intervals) else float("nan")

    ##### Post-baseline #####
    # Post-baseline: compute respiratory rate (RR; in bpm) and mean respiratory cycle duration
    post = mb["post_baseline"]
    post_peaks = post.get("exp_onsets", [])
    post_troughs = post.get("insp_onsets", [])
    post_ms = post["start_end_idx"][1] - post["start_end_idx"][0]

    post_cycles = min(len(post_peaks), len(post_troughs))
    rr_bpm_post = round((post_cycles * 60) / (post_ms / 1000), 2) if post_cycles > 0 else float("nan")

    # Calculate respiratory cycle duration
    if len(post_peaks) > 1:
        post_intervals = np.diff(post_peaks)
    elif len(post_troughs) > 1:
        post_intervals = np.diff(post_troughs)
    else:
        post_intervals = []

    cycle_s_post = round(np.mean(post_intervals)/1000, 3) if len(post_intervals) else float("nan")

    # Save summary data in summ_resp
    summ_resp.append({
        "subjID": curr_subject,
        "total_exp_onsets": tot_exp_onsets, "total_insp_onsets": tot_insp_onsets, "total_resp_onsets": tot_resp_onsets,
        "manualcorr_resp_onsets": n_corr_total, "pct_manualcorr": pct_manualcorr_resp,
        "len_mainblocks_idx": tot_len_idx, "len_badsegm_idx": tot_badsegm_idx, "pct_badsegm": pct_badsegm_resp,
        "rr_bpm_pre": rr_bpm_pre, "rr_bpm_post": rr_bpm_post,
        "cycle_s_pre": cycle_s_pre, "cycle_s_post": cycle_s_post,
    })

summ_resp = pd.DataFrame(summ_resp)


# Define directory of storage of summary RESP data and save TSV file 
summ_df_fname = f"task-{exp_name}_resp-mainblocks_summary.tsv"
summ_df_path =  os.path.join(wd, "derivatives", "resp-preproc", summ_df_fname)
summ_resp.to_csv(summ_df_path, index=False, sep='\t', na_rep="n/a")

In [36]:
# Group-level descriptives: RESP data quality
md("On average, {}% of all expiration/inspiration onsets were manually corrected in the main blocks (i.e., baseline and experimental task, excluding breaks and instructions). On average, {}% of the RESP recording was annotated as being noisy or low-quality.".format(
    round(summ_resp['pct_manualcorr'].mean(),2), round(summ_resp['pct_badsegm'].mean(),2)
))

On average, 1.6% of all expiration/inspiration onsets were manually corrected in the main blocks (i.e., baseline and experimental task, excluding breaks and instructions). On average, 9.41% of the RESP recording was annotated as being noisy or low-quality.

In [37]:
# Group-level descriptives: respiratory rate and cycle duration at rest
md("Mean respiratory rate at rest, averaged across pre- and post-baseline, varied from {} to {} breaths per min (M={}, SD={}), with mean respiratory cycles ranging from {} to {} s (M={}, SD={}).".format(
    round(summ_resp[['rr_bpm_pre', 'rr_bpm_post']].min().min(),2), round(summ_resp[['rr_bpm_pre', 'rr_bpm_post']].max().max(),2), 
    round(((summ_resp['rr_bpm_pre'].mean() + summ_resp['rr_bpm_post'].mean())/2),2), 
    round(((summ_resp['rr_bpm_pre'].std() + summ_resp['rr_bpm_post'].std())/2),2), 
    round(summ_resp[['cycle_s_pre', 'cycle_s_post']].min().min(),2), round(summ_resp[['cycle_s_pre', 'cycle_s_post']].max().max(),2), 
    round(((summ_resp['cycle_s_pre'].mean() + summ_resp['cycle_s_post'].mean())/2),2), 
    round(((summ_resp['cycle_s_pre'].std() + summ_resp['cycle_s_post'].std())/2),2)
))

Mean respiratory rate at rest, averaged across pre- and post-baseline, varied from 7.0 to 19.67 breaths per min (M=14.0, SD=2.75), with mean respiratory cycles ranging from 3.03 to 8.49 s (M=4.46, SD=1.04).